# Analysis 1 — Profiling TFU users inside the Gifter and Follow-Bettor pools

**Question:** within each month, of all Gifters (any gift) and all Follow Bettors (any follow-bet),
how many are TFU, and what do those TFU users look like?

- **Gifter pool** = `total_tip_count + total_box_count + total_wheel_count > 0`
- **Follow Bettor pool** = `total_follow_bet_count > 0`
- **TFU** = sits in **both** pools (gift ✓ AND follow-bet ✓), i.e. the intersection

All analysis is **single-month** — no next-month join, no conversion rate.

Sections:
1. Population overview — pool sizes, TFU count and share within each pool, 6-month trend
2. Segment distribution of TFU users (categorical: account age, watch bucket, device, etc.)
3. Behavioral profile of TFU users (numeric: watch, chat, gifting, betting, breadth)
4. Correlation heatmaps — numeric features vs `is_tfu`, within each pool
5. Monthly trends — how TFU share within each pool moves over 6 months

## 0. Setup

In [ ]:
# --- Colab auth + BigQuery client ---
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')

# TODO: fill in your project / dataset
PROJECT_ID = 'nf-bifrost'
DATASET    = 'your_dataset'                       # <-- update
TABLE      = f'{PROJECT_ID}.{DATASET}.tfu_user_monthly'

client = bigquery.Client(project=PROJECT_ID)

## 1. Pull `tfu_user_monthly` and define pools

Schema columns:
- **IDs/time:** `cust_id`, `data_month`, `month_year`, `month_end`
- **Account:** `account_age_days`, `account_age_tier`, `site`, `currency`
- **Loyalty:** `sessions_count`, `distinct_streamers`, `sessions_bucket`
- **Watch:** `total_watch_sec`, `avg_watch_sec_per_session`, `watch_bucket`
- **Chat:** `total_messages`, `chat_sessions`, `total_bullet_sec`, `total_chatroom_sec`
- **Gifting:** `total_tip_count/usd`, `total_box_count/usd`, `total_wheel_count/usd`
- **Betting:** `total_bet_count`, `total_member_to`, `total_bdw_bet_count`, `total_follow_bet_count`
- **Preferences:** `stream_type_pref`, `device_pref`, `time_segment`, `day_segment`, `breadth_score`
- **Streamer affinity:** `top_follow_streamer*`, `top_gift_streamer*`, `top_streamer_is_same`
- **Target:** `is_tfu`

Pools are **broad/overlapping**: TFU users belong to **both** the Gifter and Follow Bettor pools.

In [ ]:
sql = f"""
SELECT *
FROM `{TABLE}`
"""
df = client.query(sql).to_dataframe()
df['data_month'] = pd.to_datetime(df['data_month'])
print('rows:', len(df), ' users:', df.cust_id.nunique(),
      ' months:', sorted(df.data_month.dt.strftime('%Y-%m').unique()))

In [ ]:
# --- Pool flags (broad / overlapping) ---
df['has_gift'] = ((df['total_tip_count'].fillna(0)
                   + df['total_box_count'].fillna(0)
                   + df['total_wheel_count'].fillna(0)) > 0).astype(int)

df['has_follow_bet'] = (df['total_follow_bet_count'].fillna(0) > 0).astype(int)

# Sanity check vs is_tfu in the table
df['is_tfu_check'] = ((df['has_gift'] == 1) & (df['has_follow_bet'] == 1)).astype(int)
mismatch = (df['is_tfu'] != df['is_tfu_check']).sum()
print(f'is_tfu sanity mismatches (should be 0): {mismatch}')

# Convenience subframes
gifter  = df[df['has_gift']       == 1].copy()
fbettor = df[df['has_follow_bet'] == 1].copy()
tfu     = df[df['is_tfu']         == 1].copy()

print(f'Gifter pool rows : {len(gifter):,}')
print(f'Follow Bettor pool rows : {len(fbettor):,}')
print(f'TFU rows (intersection) : {len(tfu):,}')

## Section 1 — Population overview

Per month:
- Gifter pool size, Follow Bettor pool size, TFU count
- TFU share **within the Gifter pool** = TFU / Gifters
- TFU share **within the Follow Bettor pool** = TFU / Follow Bettors

In [ ]:
overview = (df.groupby('data_month')
              .agg(gifter_pool    = ('has_gift', 'sum'),
                   fbettor_pool   = ('has_follow_bet', 'sum'),
                   tfu_users      = ('is_tfu', 'sum'))
              .reset_index())
overview['tfu_share_of_gifters']  = overview['tfu_users'] / overview['gifter_pool']
overview['tfu_share_of_fbettors'] = overview['tfu_users'] / overview['fbettor_pool']
overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

size_long = overview.melt(id_vars='data_month',
                          value_vars=['gifter_pool', 'fbettor_pool', 'tfu_users'],
                          var_name='pool', value_name='users')
sns.barplot(data=size_long, x='data_month', y='users', hue='pool', ax=axes[0])
axes[0].set_title('Pool sizes per month')
axes[0].tick_params(axis='x', rotation=45)

share_long = overview.melt(id_vars='data_month',
                           value_vars=['tfu_share_of_gifters', 'tfu_share_of_fbettors'],
                           var_name='pool', value_name='tfu_share')
sns.lineplot(data=share_long, x='data_month', y='tfu_share', hue='pool',
             marker='o', linewidth=2, ax=axes[1])
axes[1].set_title('TFU share within each pool')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

## Section 2 — Segment distribution of TFU users

For each categorical column, compare three groups:
- **TFU** — in both pools
- **Gifter, non-TFU** — gifted but did not follow-bet
- **Follow Bettor, non-TFU** — follow-bet but did not gift

Each bar shows the share of that group falling into the category — so we can see whether TFU users skew older/younger, heavier/lighter watchers, mobile/web, etc.

In [ ]:
df['group'] = np.select(
    [df['is_tfu'] == 1,
     (df['has_gift'] == 1) & (df['has_follow_bet'] == 0),
     (df['has_follow_bet'] == 1) & (df['has_gift'] == 0)],
    ['TFU', 'Gifter, non-TFU', 'Follow Bettor, non-TFU'],
    default='Other'
)

group_order = ['TFU', 'Gifter, non-TFU', 'Follow Bettor, non-TFU']
df['group'].value_counts()

In [ ]:
categorical_cols = [
    'account_age_tier', 'watch_bucket', 'sessions_bucket',
    'time_segment', 'day_segment',
    'stream_type_pref', 'device_pref',
]

def share_within_group(frame, col):
    plot_df = frame[frame['group'].isin(group_order)].copy()
    t = (plot_df.groupby(['group', col]).size()
                 .groupby(level=0).apply(lambda s: s / s.sum())
                 .rename('share').reset_index())
    return t

for col in categorical_cols:
    t = share_within_group(df, col)
    plt.figure(figsize=(9, 3.8))
    sns.barplot(data=t, x=col, y='share', hue='group', hue_order=group_order)
    plt.title(f'{col} — share within each group')
    plt.xticks(rotation=30, ha='right')
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.tight_layout(); plt.show()

In [ ]:
# Headline: TFU-only distributions as compact tables
for col in categorical_cols:
    print(f'--- TFU share by {col} ---')
    print((tfu[col].value_counts(normalize=True)
                   .mul(100).round(1).astype(str) + '%').to_string())
    print()

## Section 3 — Behavioral profile of TFU users

Compare TFU vs non-TFU on numeric behavior — watch, chat, gifting depth, betting, breadth. Same three groups as Section 2.

In [ ]:
numeric_features = [
    # Watch
    'total_watch_sec', 'avg_watch_sec_per_session',
    # Chat
    'total_messages', 'chat_sessions', 'total_bullet_sec', 'total_chatroom_sec',
    # Gifting
    'total_tip_count', 'total_box_count', 'total_wheel_count',
    'total_tip_usd', 'total_box_usd', 'total_wheel_usd',
    # Betting
    'total_bet_count', 'total_member_to', 'total_bdw_bet_count', 'total_follow_bet_count',
    # Breadth / loyalty
    'breadth_score', 'sessions_count', 'distinct_streamers',
]

df['total_gift_usd'] = (df['total_tip_usd'].fillna(0)
                        + df['total_box_usd'].fillna(0)
                        + df['total_wheel_usd'].fillna(0))
numeric_features.append('total_gift_usd')

In [ ]:
profile = (df[df['group'].isin(group_order)]
             .groupby('group')[numeric_features]
             .agg(['median', 'mean'])
             .round(2))
profile.loc[group_order]

In [ ]:
# Boxplots (log scale) per numeric feature
plot_df = df[df['group'].isin(group_order)].copy()

n = len(numeric_features)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    ax = axes[i]
    data = plot_df[[feat, 'group']].copy()
    data[feat] = data[feat].clip(lower=0) + 1
    sns.boxplot(data=data, x='group', y=feat, order=group_order,
                ax=ax, showfliers=False)
    ax.set_yscale('log')
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout(); plt.show()

## Section 4 — Correlation heatmaps

Within each pool (Gifter, Follow Bettor), Spearman correlation between numeric features and `is_tfu`. Highlights which behaviors are most associated with being TFU **inside that pool**.

In [ ]:
def corr_heatmap(frame, pool_name):
    cols = numeric_features + ['is_tfu']
    corr = frame[cols].corr(method='spearman')
    plt.figure(figsize=(11, 9))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
                annot=False, square=False)
    plt.title(f'Spearman correlation — {pool_name} (n={len(frame)})')
    plt.tight_layout(); plt.show()

    top = (corr['is_tfu'].drop('is_tfu')
                          .abs().sort_values(ascending=False)
                          .head(15))
    print(f'\nTop |corr| with is_tfu — {pool_name}:')
    print(top.round(3))

corr_heatmap(gifter,  'Gifter pool')
corr_heatmap(fbettor, 'Follow Bettor pool')

## Section 5 — Monthly trends

How TFU users move over 6 months: absolute count, share within each pool, and any drift in their categorical mix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(data=overview, x='data_month', y='tfu_users',
             marker='o', linewidth=2, ax=axes[0])
axes[0].set_title('TFU user count — 6-month trend')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=share_long, x='data_month', y='tfu_share', hue='pool',
             marker='o', linewidth=2, ax=axes[1])
axes[1].set_title('TFU share within each pool — 6-month trend')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

In [ ]:
# Categorical mix drift: TFU-only share of each category per month
for col in categorical_cols:
    t = (tfu.groupby(['data_month', col]).size()
             .groupby(level=0).apply(lambda s: s / s.sum())
             .rename('share').reset_index())
    plt.figure(figsize=(9, 3.5))
    sns.lineplot(data=t, x='data_month', y='share', hue=col, marker='o')
    plt.title(f'TFU mix over time — {col}')
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.xticks(rotation=45)
    plt.tight_layout(); plt.show()

## Takeaways scratch-pad

Fill in after running:
- Avg TFU share of Gifter pool ≈ _ ;  of Follow Bettor pool ≈ _
- TFU users skew toward which `account_age_tier`? _
- TFU users skew toward which `watch_bucket`? _
- Device / time / day-segment skew: _
- Biggest numeric gaps (TFU vs non-TFU): _
- Top features correlated with `is_tfu` in Gifter pool: _
- Top features correlated with `is_tfu` in Follow Bettor pool: _
- 6-month trend: TFU share rising / flat / falling: _